# psfFlux vs scienceFlux — raw flux error plots (no normalisation)

## Purpose

This notebook displays the raw fluxes in nJy:

- `psfFlux`     (difference-image PSF flux, can be negative)
- `scienceFlux` (PSF flux measured on the science calexp, always positive for a real source)

**No normalisation / ratio** is applied.  The goal is to directly compare the
absolute flux levels and their measurement uncertainties.

Two sets of 7-panel figures are produced (6 bands + 1 combined):

1. **Section 6** — one figure per DIA object (top-N by alert count), with shared
   x and y scales across band panels.

2. **Section 7** — one figure per band (+ combined), accumulating all top-N
   objects on the same axes (different colours per object).

---

- **Author:** Sylvie Dagoret-Campagne  
- **Affiliation:** IJCLab / IN2P3 / CNRS  
- **Follows:** `06_relativeFlux_science.ipynb`
- **Creation date:** 2026-04-07  
- **Last update:** 2026-04-12
- **Subject:** Fink alert broker — Rubin LSST DIA photometry diagnostics

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from astropy.time import Time

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
# ── Notebook tag & paths ──────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_07"
DIR_DATA_IN = "data_FINK_BLOCK_LC_03"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)

FILE_BUTLER = os.path.join(DIR_DATA_IN, "src_joined_butler.parquet")
FILE_CONSDB = os.path.join(DIR_DATA_IN, "src_joined_consdb.parquet")

N_TOP = 20  # top-N objects by alert count

BANDS = list("ugrizy")
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

# ── Column names ──────────────────────────────────────────────────────────────
COL_OBJ = "r:diaObjectId"
COL_SRC = "r:diaSourceId"
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"
COL_PSF = "r:psfFlux"
COL_PSFERR = "r:psfFluxErr"

SCIENCE_FLUX_CANDIDATES = [
    "r:scienceFlux",
    "r:psfScience",
    "r:psfScienceFlux",
    "scienceFlux",
    "psfScience",
    "psfScienceFlux",
]
SCIENCE_ERR_CANDIDATES = [
    "r:scienceSigma",
    "r:scienceFluxErr",
    "r:psfScienceSigma",
    "r:psfScienceFluxErr",
    "scienceSigma",
    "scienceFluxErr",
    "psfScienceSigma",
]

TEMPLATE_FLUX_CANDIDATES = [
    "psfTemplateFlux",
    "r:templateFlux",
    "r:template",
    "r:psfScienceFlux",
    "templateFlux",
    "template",
    "psfTemplateFlux",
]
TEMPLATE_ERR_CANDIDATES = [
    "r:templaeSigma",
    "r:templateFluxErr",
    "r:psfTemplateSigma",
    "r:psfTemplateFluxErr",
    "templateSigma",
    "templateFluxErr",
    "psfTemplateSigma",
]


def savefig(name):
    """Save current figure as both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print(f"Input directory  : {os.path.abspath(DIR_DATA_IN)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"N_TOP            : {N_TOP}")

## 2. Load data

In [ ]:
if os.path.exists(FILE_BUTLER):
    df = pd.read_parquet(FILE_BUTLER)
    src_label = "butler"
    print(f"Loaded butler file: {FILE_BUTLER}")
elif os.path.exists(FILE_CONSDB):
    df = pd.read_parquet(FILE_CONSDB)
    src_label = "consdb"
    print(f"Butler file not found. Loaded consDb file: {FILE_CONSDB}")
else:
    raise FileNotFoundError(
        f"Neither {FILE_BUTLER} nor {FILE_CONSDB} found.\n"
        "Run notebook 03_associateFinkAlerts-RubinVisits.ipynb first."
    )

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Source label: {src_label}")

## 3. Schema inspection — detect scienceFlux column

In [ ]:
COL_SCI = None
COL_SCIERR = None

for candidate in SCIENCE_FLUX_CANDIDATES:
    if candidate in df.columns:
        COL_SCI = candidate
        print(f"Found science flux column  : '{COL_SCI}'")
        break

for candidate in SCIENCE_ERR_CANDIDATES:
    if candidate in df.columns:
        COL_SCIERR = candidate
        print(f"Found science flux err col : '{COL_SCIERR}'")
        break

if COL_SCI is None:
    print("WARNING: No scienceFlux-like column found.")
    sci_cols = [c for c in df.columns if "sci" in c.lower()]
    print(f"Columns with 'sci': {sci_cols}")
else:
    n_valid = df[COL_SCI].notna().sum()
    print(f"Non-NaN values: {n_valid:,} / {len(df):,}  ({100 * n_valid / len(df):.1f}%)")

has_science = COL_SCI is not None
has_sci_err = COL_SCIERR is not None

print(f"\n  psfFlux    : {COL_PSF}")
print(f"  scienceFlux: {COL_SCI}")
print(f"  scienceErr : {COL_SCIERR}")

In [ ]:
COL_TEMP = None
COL_TEMPERR = None

for candidate in TEMPLATE_FLUX_CANDIDATES:
    if candidate in df.columns:
        COL_TEMP = candidate
        print(f"Found template flux column  : '{COL_TEMP}'")
        break

for candidate in TEMPLATE_ERR_CANDIDATES:
    if candidate in df.columns:
        COL_TEMPERR = candidate
        print(f"Found template flux err col : '{COL_TEMPERR}'")
        break

if COL_TEMP is None:
    print("WARNING: No templateFlux-like column found.")
    temp_cols = [c for c in df.columns if "temp" in c.lower()]
    print(f"Columns with 'temp': {temp_cols}")
else:
    n_valid = df[COL_TEMP].notna().sum()
    print(f"Non-NaN values: {n_valid:,} / {len(df):,}  ({100 * n_valid / len(df):.1f}%)")

has_template = COL_TEMP is not None
has_template_err = COL_TEMPERR is not None

print(f"\n  psfFlux    : {COL_PSF}")
print(f"  scienceFlux: {COL_SCI}")
print(f"  scienceErr : {COL_SCIERR}")
print(f"  templateFlux: {COL_TEMP}")
print(f"  templateErr : {COL_TEMPERR}")

## 4. Rank DIA objects by number of alerts

In [ ]:
alert_counts = df.groupby(COL_OBJ).size().rename("n_alerts").sort_values(ascending=False)
print(f"Total unique DIA objects : {len(alert_counts):,}")
print(f"Top {N_TOP}:")
print(alert_counts.head(N_TOP).to_string())

In [ ]:
top_objects = alert_counts.head(N_TOP).index.tolist()
df_top = df[df[COL_OBJ].isin(top_objects)].copy()
df_top[COL_MJD] = df_top[COL_MJD].astype(float)
print(f"Rows kept: {len(df_top):,}")

In [ ]:
df_top.head()

## 5. Utility functions

In [ ]:
AB_FLUX_ZERO = 3631e9  # nJy at AB zero-point


def flux_to_mag(flux_nJy, flux_err_nJy=None):
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def mjd_to_datestr(mjd_array):
    """Convert MJD array to ISO date strings 'YYYY-MM-DD'."""
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def _add_date_axis(ax, dt_array, t0_mjd):
    """Secondary x-axis on top of *ax* showing calendar dates."""
    dt_finite = dt_array[np.isfinite(dt_array)]
    if len(dt_finite) == 0:
        return
    dt_min, dt_max = float(dt_finite.min()), float(dt_finite.max())
    if dt_max <= dt_min:
        tick_dt = np.array([dt_min])
    else:
        n_ticks = min(5, max(2, int((dt_max - dt_min) / 10) + 2))
        tick_dt = np.linspace(dt_min, dt_max, n_ticks)
    tick_dates = mjd_to_datestr(t0_mjd + tick_dt)
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(tick_dt)
    ax2.set_xticklabels(tick_dates, rotation=35, ha="left", fontsize=7)
    ax2.tick_params(axis="x", length=3)


def _shared_lim(arrays, margin=0.05):
    """
    Compute a common [vmin, vmax] range from a list of arrays,
    adding a fractional margin.  Returns None if no finite data.
    """
    all_vals = np.concatenate([np.asarray(a, dtype=float).ravel() for a in arrays])
    finite = all_vals[np.isfinite(all_vals)]
    if len(finite) == 0:
        return None
    vmin, vmax = finite.min(), finite.max()
    span = max(vmax - vmin, 1e-6)
    return vmin - margin * span, vmax + margin * span


def _object_color_palette(n_objects):
    """
    Return a list of n_objects distinguishable colours drawn from tab20
    (or tab10 if n_objects <= 10).
    """
    cmap = cm.get_cmap("tab20" if n_objects > 10 else "tab10")
    return [cmap(i / max(n_objects - 1, 1)) for i in range(n_objects)]


print("Utility functions defined.")

## 5b. Per-object pixel/flux table — `get_diaobject_table()`

This function extracts a **flat, analysis-ready DataFrame** for a single `diaObjectId`
from the already-loaded DataFrame `df` (the parquet file from Section 2).

It uses the column names resolved in Section 3 (`COL_SCI`, `COL_TEMP`, etc.) and
returns a clean table with standardised short column names.

| column | source column | description |
|---|---|---|
| `mjd` | `r:midpointMjdTai` | Midpoint of exposure (TAI, MJD) |
| `band` | `r:band` | Photometric band (`u/g/r/i/z/y`) |
| `visit` | `r:visit` | Butler visitId (Int64, nullable) |
| `detector` | `r:detector` | CCD detector number (Int64, nullable) |
| `x` | `r:x` | Centroid x pixel coordinate on CCD |
| `y` | `r:y` | Centroid y pixel coordinate on CCD |
| `psfFlux` | `r:psfFlux` | PSF flux of the difference-image source (nJy) |
| `psfFluxErr` | `r:psfFluxErr` | Uncertainty on psfFlux (nJy) |
| `scienceFlux` | `COL_SCI` | Flux on the science image (nJy); NaN if unavailable |
| `scienceFluxErr` | `COL_SCIERR` | Uncertainty on scienceFlux (nJy); NaN if unavailable |
| `templateFlux` | `COL_TEMP` | Flux on the template image (nJy); NaN if unavailable |
| `templateFluxErr` | `COL_TEMPERR` | Uncertainty on templateFlux (nJy); NaN if unavailable |

The function works on `df` by default but accepts any compatible DataFrame via
the optional `source_df` argument (useful to pass `df_top` for a pre-filtered subset).

In [ ]:
def get_diaobject_table(diaObjectId, source_df=None) -> pd.DataFrame:
    """
    Return a flat analysis-ready DataFrame for a single diaObjectId.

    Extracts rows from the parquet DataFrame loaded in Section 2 and
    renames columns to short, standardised names.

    Parameters
    ----------
    diaObjectId : int or str
        Target diaObjectId to extract.
    source_df : pd.DataFrame or None
        DataFrame to query.  Defaults to the module-level `df`.
        Pass `df_top` to work on the pre-filtered top-N subset.

    Returns
    -------
    pd.DataFrame
        Columns: mjd, band, visit (Int64), detector (Int64), x, y,
                 psfFlux, psfFluxErr,
                 scienceFlux, scienceFluxErr,
                 templateFlux, templateFluxErr
        Sorted by (mjd, band).  Empty DataFrame if object not found.

    Notes
    -----
    - COL_SCI, COL_SCIERR, COL_TEMP, COL_TEMPERR are resolved in Section 3.
      If a column was not found, the corresponding output column is filled with NaN.
    - All flux columns are in nJy.
    - r:visit and r:detector are cast to nullable Int64.
    """
    _src = source_df if source_df is not None else df

    # ── Filter to requested object ────────────────────────────────────────────
    mask = _src[COL_OBJ].astype(str) == str(diaObjectId)
    df_obj = _src[mask].copy()

    if df_obj.empty:
        return pd.DataFrame(
            columns=[
                "diaObj",
                "diaSrc",
                "mjd",
                "band",
                "visit",
                "detector",
                "x",
                "y",
                "psfFlux",
                "psfFluxErr",
                "scienceFlux",
                "scienceFluxErr",
                "templateFlux",
                "templateFluxErr",
                "day_obs",
                "day_month",
            ]
        )

    # ── Build output DataFrame column by column ───────────────────────────────
    out = pd.DataFrame()

    # Mandatory columns
    out["diaObj"] = df_obj[COL_OBJ].values
    out["diaSrc"] = df_obj[COL_SRC].values
    out["mjd"] = pd.to_numeric(df_obj[COL_MJD], errors="coerce")
    out["band"] = df_obj[COL_BAND].values
    out["psfFlux"] = pd.to_numeric(df_obj[COL_PSF], errors="coerce")
    out["psfFluxErr"] = pd.to_numeric(df_obj[COL_PSFERR], errors="coerce")

    # visit / detector — present in butler-joined parquet, nullable Int64
    for col_in, col_out in (("r:visit", "visit"), ("r:detector", "detector")):
        if col_in in df_obj.columns:
            out[col_out] = pd.to_numeric(df_obj[col_in], errors="coerce").astype("Int64")
        else:
            out[col_out] = pd.array([pd.NA] * len(df_obj), dtype="Int64")

    # x, y pixel coordinates
    for col_in, col_out in (("r:x", "x"), ("r:y", "y")):
        if col_in in df_obj.columns:
            out[col_out] = pd.to_numeric(df_obj[col_in], errors="coerce")
        else:
            out[col_out] = np.nan

    # scienceFlux / scienceFluxErr
    if COL_SCI is not None and COL_SCI in df_obj.columns:
        out["scienceFlux"] = pd.to_numeric(df_obj[COL_SCI], errors="coerce")
    else:
        out["scienceFlux"] = np.nan

    if COL_SCIERR is not None and COL_SCIERR in df_obj.columns:
        out["scienceFluxErr"] = pd.to_numeric(df_obj[COL_SCIERR], errors="coerce")
    else:
        out["scienceFluxErr"] = np.nan

    # templateFlux / templateFluxErr
    if COL_TEMP is not None and COL_TEMP in df_obj.columns:
        out["templateFlux"] = pd.to_numeric(df_obj[COL_TEMP], errors="coerce")
    else:
        out["templateFlux"] = np.nan

    if COL_TEMPERR is not None and COL_TEMPERR in df_obj.columns:
        out["templateFluxErr"] = pd.to_numeric(df_obj[COL_TEMPERR], errors="coerce")
    else:
        out["templateFluxErr"] = np.nan

    # ── Final column order and sort ───────────────────────────────────────────
    out = (
        out[
            [
                "diaObj",
                "diaSrc",
                "mjd",
                "band",
                "visit",
                "detector",
                "x",
                "y",
                "psfFlux",
                "psfFluxErr",
                "scienceFlux",
                "scienceFluxErr",
                "templateFlux",
                "templateFluxErr",
            ]
        ]
        .sort_values(["mjd", "band"], na_position="last")
        .reset_index(drop=True)
    )
    out["day_obs"] = out["visit"] // 100000
    out["month_obs"] = out["visit"] // 10000000

    return out


print("get_diaobject_table() defined.")
print("Usage:")
print("  df_obj = get_diaobject_table(diaObjectId)")
print("  df_obj = get_diaobject_table(diaObjectId, source_df=df_top)")
print("  Columns: mjd, band, visit, detector, x, y,")
print("           psfFlux, psfFluxErr,")
print("           scienceFlux, scienceFluxErr,")
print("           templateFlux, templateFluxErr")

### 5b-demo — Quick test of `get_diaobject_table()`

Uses the first object from `top_objects` (requires Section 4 to have been run).

In [ ]:
# ── Quick demo on the highest-alert object ────────────────────────────────────
# Requires Section 4 to have been run (top_objects must be defined).

# Uncomment the lines below to run the test:
_test_oid = top_objects[1]
df_test = get_diaobject_table(_test_oid)
print(f"diaObjectId {_test_oid}  →  {len(df_test)} rows")
print(f"  bands present   : {sorted(df_test['band'].dropna().unique())}")
print(f"  visit range     : {df_test['visit'].dropna().min()} … {df_test['visit'].dropna().max()}")
print(f"  mjd  range      : {df_test['mjd'].min():.2f} … {df_test['mjd'].max():.2f}")
print(f"  scienceFlux NaN : {df_test['scienceFlux'].isna().sum()} / {len(df_test)}")
print(f"  templateFlux NaN: {df_test['templateFlux'].isna().sum()} / {len(df_test)}")

# display(df_test.head(10))
display(df_test)

print("Demo cell ready — uncomment and run after Section 4.")

In [ ]:
df_test[df_test["day_obs"] == 20260226]

## 5b Load diaObject catalog  from file

In [ ]:
DIR_CAT = "data_FINK_BLOCK_LC_01"
_cat_priority = [
    os.path.join(DIR_CAT, "diaobj_catalogue_gaia_stable.csv"),
    os.path.join(DIR_CAT, "diaobj_catalogue.csv"),
]
df_cat_ref = None
for _cat_path in _cat_priority:
    if os.path.exists(_cat_path):
        df_cat_ref = pd.read_csv(_cat_path, low_memory=False)
        df_cat_ref["diaObjectId"] = df_cat_ref["diaObjectId"].astype(str)
        break

## 6. Per-object: psfFlux vs scienceFlux error plots

For each DIA object: **7 subplots** (one per band `u g r i z y` + combined).

Each panel shows:
- **Filled circles** `●` = `scienceFlux` ± `scienceFluxErr`  (blue-ish)
- **Open squares**   `□` = `psfFlux`     ± `psfFluxErr`       (orange-ish, same band colour)

x-axis : Δt [days] from first alert of the object.  
y-axis : flux [nJy] — **no normalisation**.  
Shared x and y scales across the 6 band panels of each figure.

In [ ]:
def plot_psf_vs_sci_one_object(obj_id, df_obj, src_label, rank):
    """
    Plot psfFlux vs scienceFlux in nJy for one DIA object.

    Layout: 7 panels (u g r i z y | all-bands).
    Shared x-axis and y-axis across band panels (indices 0-5).
    """

    # special with object catalog for flux straight-lines
    oid_str = str(obj_id)
    ref_psf = {}
    ref_sci = {}
    if df_cat_ref is not None:
        mask_cat = df_cat_ref["diaObjectId"].astype(str) == oid_str
        if mask_cat.any():
            cat_row = df_cat_ref[mask_cat].iloc[0]
            for b in BANDS:
                for dest, col in [(ref_psf, f"r:{b}_psfFluxMean"), (ref_sci, f"r:{b}_scienceFluxMean")]:
                    try:
                        fval = float(cat_row[col])
                        dest[b] = fval if np.isfinite(fval) else None
                    except (KeyError, TypeError, ValueError):
                        dest[b] = None

    # usual  way to
    t0_mjd = df_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]
    field = df_obj["field"].iloc[0] if "field" in df_obj.columns else ""
    n_subplots = len(BANDS) + 1

    fig, axes = plt.subplots(
        1,
        n_subplots,
        figsize=(3.2 * n_subplots, 4.5),
        constrained_layout=True,
    )

    # ── Pre-compute data and global axis limits ───────────────────────────────
    band_data = {}  # band → (dt, psf, psf_err, sci, sci_err)
    all_flux = []  # all flux values (psf + sci) for shared y-axis
    all_dt = []  # all Δt values for shared x-axis

    for band in BANDS:
        mask = df_obj[COL_BAND] == band
        df_b = df_obj[mask].sort_values(COL_MJD)
        if len(df_b) == 0:
            band_data[band] = None
            continue
        dt = df_b[COL_MJD].values - t0_mjd
        psf = df_b[COL_PSF].values.astype(float)
        psf_err = df_b[COL_PSFERR].values.astype(float)
        sci = df_b[COL_SCI].values.astype(float) if has_science else None
        sci_err = df_b[COL_SCIERR].values.astype(float) if has_sci_err else None

        temp = df_b[COL_TEMP].values.astype(float) if has_template else None
        temp_err = df_b[COL_TEMPERR].values.astype(float) if has_template_err else None

        band_data[band] = (dt, psf, psf_err, sci, sci_err, temp, temp_err)

        all_dt.append(dt)
        all_flux.append(psf)
        if sci is not None:
            all_flux.append(sci)

    ylim = _shared_lim(all_flux) if all_flux else None
    xlim = _shared_lim(all_dt) if all_dt else None

    # ── Per-band panels ───────────────────────────────────────────────────────
    for idx, band in enumerate(BANDS):
        ax = axes[idx]
        color = BAND_COLORS.get(band, "k")

        if band_data[band] is None:
            ax.text(
                0.5,
                0.5,
                "no data",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="gray",
                fontsize=8,
            )
            ax.set_title(f"band {band}", fontsize=9)
            if xlim:
                ax.set_xlim(xlim)
            if ylim:
                ax.set_ylim(ylim)
            ax.set_xlabel("Δt [days]", fontsize=8)
            continue

        dt, psf, psf_err, sci, sci_err, temp, temp_err = band_data[band]

        # scienceFlux — filled circles
        if sci is not None:
            ax.errorbar(
                dt,
                sci,
                yerr=sci_err,
                fmt="o",
                ms=5,
                color=color,
                ecolor=color,
                elinewidth=0.9,
                capsize=2,
                alpha=0.90,
                label="scienceFlux",
            )

        # psfFlux — open squares
        ax.errorbar(
            dt,
            psf,
            yerr=psf_err,
            fmt="s",
            ms=4,
            color=color,
            ecolor=color,
            elinewidth=0.7,
            capsize=2,
            alpha=0.55,
            mfc="none",
            label="psfFlux",
        )

        # templates — open squares
        ax.errorbar(
            dt,
            temp,
            yerr=temp_err,
            fmt="+",
            ms=4,
            color=color,
            ecolor=color,
            elinewidth=0.7,
            capsize=2,
            alpha=0.55,
            mfc="none",
            label="templateFlux",
        )

        # add object flux per band
        v_sci = ref_sci.get(band)
        if v_sci is not None:
            mag, merr = flux_to_mag(v_sci)
            ax.axhline(
                v_sci, color=color, lw=1.2, ls="-.", alpha=0.85, label=f"obj sciFluxMean m={mag:.1f} mag"
            )
        v_psf = ref_psf.get(band)
        if v_psf is not None:
            mag, merr = flux_to_mag(v_psf)
            ax.axhline(
                v_psf, color=color, lw=1.2, ls=":", alpha=0.85, label=f"obj psfFluxMean m={mag:.1f} mag"
            )

        ax.axhline(0.0, color="k", lw=0.6, ls=":", alpha=0.4)
        ax.set_title(f"band {band}", fontsize=9)
        ax.set_ylabel("flux [nJy]", fontsize=8)
        ax.set_xlabel("Δt [days]", fontsize=8)
        ax.legend(fontsize=7, loc="best")
        if xlim:
            ax.set_xlim(xlim)
        if ylim:
            ax.set_ylim(ylim)
        _add_date_axis(ax, dt, t0_mjd)

    # ── Combined panel ────────────────────────────────────────────────────────
    ax_all = axes[-1]
    for band in BANDS:
        if band_data[band] is None:
            continue
        dt, psf, psf_err, sci, sci_err, temp, temp_err = band_data[band]
        color = BAND_COLORS.get(band, "k")
        if sci is not None:
            ax_all.errorbar(
                dt,
                sci,
                yerr=sci_err,
                fmt="o",
                ms=3,
                color=color,
                ecolor=color,
                elinewidth=0.7,
                capsize=2,
                alpha=0.80,
                label=f"{band} sci",
            )
        ax_all.errorbar(
            dt,
            psf,
            yerr=psf_err,
            fmt="s",
            ms=3,
            color=color,
            ecolor=color,
            elinewidth=0.6,
            capsize=2,
            alpha=0.45,
            mfc="none",
            label=f"{band} psf",
        )

        if temp is not None:
            ax_all.errorbar(
                dt,
                temp,
                yerr=temp_err,
                fmt="+",
                ms=3,
                color=color,
                ecolor=color,
                elinewidth=0.7,
                capsize=2,
                alpha=0.80,
                mfc="none",
                label=f"{band} temp",
            )

        # horizontal lines with object flux
        v_sci = ref_sci.get(band)
        if v_sci is not None:
            ax_all.axhline(v_sci, color=color, lw=0.7, ls="-.", alpha=0.45, label="_nolegend_")
        v_psf = ref_psf.get(band)
        if v_psf is not None:
            ax_all.axhline(v_psf, color=color, lw=0.7, ls=":", alpha=0.45, label="_nolegend_")

    ax_all.axhline(0.0, color="k", lw=0.6, ls=":", alpha=0.4)
    ax_all.set_title("all bands — sci(●) vs psf(□)", fontsize=9)
    ax_all.set_ylabel("flux [nJy]", fontsize=8)
    ax_all.set_xlabel("Δt [days]", fontsize=8)
    ax_all.legend(fontsize=6, ncol=4, loc="best")
    if xlim:
        ax_all.set_xlim(xlim)
    _add_date_axis(ax_all, df_obj[COL_MJD].values - t0_mjd, t0_mjd)

    fig.suptitle(
        f"rank #{rank + 1}  |  diaObjectId={obj_id}  |  field={field}  |"
        f"  N={len(df_obj)}  |  t₀={t0_date}\n"
        "scienceFlux (●) vs psfFlux (□)   [nJy, no normalisation]",
        fontsize=11,
        y=1.07,
    )
    savefig(f"psf_vs_sci_obj_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
    plt.show()


print("plot_psf_vs_sci_one_object() defined.")

In [ ]:
if not has_science:
    print("scienceFlux column not available — Section 6 skipped.")
else:
    for rank, obj_id in enumerate(top_objects):
        df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
        plot_psf_vs_sci_one_object(obj_id, df_obj, src_label, rank)

## 8. Summary statistics: median flux levels

For each object and band, report the median `scienceFlux` and median `psfFlux`
to assess their absolute levels.  The ratio `median(psfFlux) / median(scienceFlux)`
indicates what fraction of the science flux is "seen" in the difference image —
it should be ~0 for a stable source in a good template.

In [ ]:
records = []

for rank, obj_id in enumerate(top_objects):
    df_obj = df_top[df_top[COL_OBJ] == obj_id]
    t0_mjd = df_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]

    row = {"rank": rank + 1, "diaObjectId": obj_id, "n_alerts": len(df_obj), "t0_date": t0_date}

    for band in BANDS:
        df_b = df_obj[df_obj[COL_BAND] == band]
        if len(df_b) == 0:
            row[f"med_sci_{band}"] = np.nan
            row[f"med_psf_{band}"] = np.nan
            row[f"ratio_psf_sci_{band}"] = np.nan
            continue

        psf_vals = df_b[COL_PSF].values.astype(float)
        med_psf = float(np.nanmedian(psf_vals))
        row[f"med_psf_{band}"] = round(med_psf, 2)

        if has_science:
            sci_vals = df_b[COL_SCI].values.astype(float)
            med_sci = float(np.nanmedian(sci_vals))
            row[f"med_sci_{band}"] = round(med_sci, 2)
            row[f"ratio_psf_sci_{band}"] = round(med_psf / med_sci, 4) if med_sci != 0 else np.nan
        else:
            row[f"med_sci_{band}"] = np.nan
            row[f"ratio_psf_sci_{band}"] = np.nan

    records.append(row)

df_stats = pd.DataFrame(records)
print("Median flux levels per object and band [nJy]:")
print("  med_sci  = median(scienceFlux)")
print("  med_psf  = median(psfFlux)")
print("  ratio    = med_psf / med_sci  (should be ~0 for stable source in good template)")
print()
display(df_stats)

In [ ]:
out_path = os.path.join(DIR_FIGS, f"median_flux_stats_{src_label}.csv")
df_stats.to_csv(out_path, index=False)
print(f"Saved → {out_path}")